<a href="https://colab.research.google.com/github/kumakun2008/CIFAR-10_Comp/blob/main/kumakun2008_AlexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vision Transformer (ViT) for CIFAR-10（Resize・Ensemble・TTA 導入版）

元の `kumakun2008_AlexNet.ipynb` をベースに、以下の論文で提案された **Vision Transformer (ViT)** を導入したノートブックです。

* **論文**: Alexey Dosovitskiy et al., *"An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale"* ([arXiv:2010.11929](https://arxiv.org/abs/2010.11929))

## 主な変更点（AlexNet → ViT）
1. **事前学習済み ViT の導入（`timm`）**: ImageNet-21K 事前学習済み ViT（`ViT-B/16`, `ViT-B/32`, `ViT-L/16` など）を読み込み、CIFAR-10（10クラス）向けにファインチューニングします。
2. **入力画像のリサイズ対応**: 32×32 の CIFAR-10 画像を `IMAGE_SIZE × IMAGE_SIZE`（デフォルト 224×224、384×384 などにも変更可能）へリサイズします。解像度を変更した場合は論文 Section 3.2 に従い、事前学習済みの位置埋め込み（Positional Embedding）が自動で 2D 補間されます。
3. **パラメータを自分で調整しやすい設計**: 解像度・バッチサイズ・学習率・エポック数・データ拡張・アンサンブル構成・TTA 設定を上部の設定セルにまとめています（元コードの「エポックごとに optimizer を再初期化してしまう不具合」も修正済みです）。
4. **アンサンブル（Ensemble）**: `ENSEMBLE_CONFIGS` リストで、シード違いや ViT バリアント違い（`ViT-B/16` + `ViT-B/32` など）を自由に組み合わせて確率平均できます。
5. **TTA（Test-Time Augmentation）**: 推論時に複数ビュー（元画像・左右反転・中央ズーム・中央ズーム＋左右反転）の予測確率を平均化します。

---

データセットは [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) を使用します。
32×32 のカラー画像が 60,000 枚（訓練 50,000 枚、テスト 10,000 枚）あり、10 クラス（飛行機・自動車・鳥・猫・鹿・犬・蛙・馬・船・トラック）に分類します。

![](https://github.com/bentrevett/pytorch-image-classification/blob/master/assets/cifar10.png?raw=1)

In [ ]:
import copy
import math
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from sklearn import decomposition, manifold
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torchvision import models
from tqdm.notebook import tqdm, trange

try:
    import timm
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'timm'])
    import timm

print(f'PyTorch: {torch.__version__} | timm: {timm.__version__}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)')
else:
    print('GPU: Not available (running on CPU)')

# 1. ハイパーパラメータ・実験設定（ここを編集してチューニング）

リサイズ解像度・学習パラメータ・アンサンブル構成・TTA 設定をここに集約しています。
まずはここを書き換えながら精度や学習時間の変化を試してください。

In [ ]:
# ============================================================
# ハイパーパラメータ・実験設定（自由に編集してください）
# ============================================================
SEED = 1234
OUTPUT_DIM = 10

# --- 1. リサイズ・前処理設定 ---
# ViT論文（arXiv:2010.11929）の標準は 224（ファインチューニング時は 384 等も有効）
# ※ 各モデルの patch_size（16 や 32）で割り切れる値を指定してください
IMAGE_SIZE = 224
RESIZE_INTERPOLATION = transforms.InterpolationMode.BICUBIC  # BICUBIC / BILINEAR

# 正規化統計の選択:
#   'cifar10'   : 元の配布コードと同様に CIFAR-10 訓練データから計算した mean / std を使用
#   'vit_paper' : ViT 論文の事前学習時と同じ mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5) を使用
NORM_MODE = 'cifar10'

# 学習時のデータ拡張（32x32 段階で適用してから IMAGE_SIZE へリサイズ）
RANDOM_ROTATION = 5
HFLIP_PROB = 0.5
RANDOM_CROP_PADDING = 4  # 元コードは 2（0 で無効化）

# --- 2. データ分割・DataLoader 設定 ---
VALID_RATIO = 0.9        # 訓練 45,000 枚 / 検証 5,000 枚
BATCH_SIZE = 64          # VRAM に合わせて 32 / 64 / 128 に調整
EVAL_BATCH_SIZE = 128    # 推論時のバッチサイズ
NUM_WORKERS = 2

# --- 3. 学習パラメータ（共通デフォルト：下の ENSEMBLE_CONFIGS で個別上書き可能）---
EPOCHS = 5               # 事前学習済みモデルのファインチューニングは少なめから試すのがおすすめ
LR = 3e-5                # ViT ファインチューニングの目安: 1e-5 ～ 1e-4
WEIGHT_DECAY = 1e-2      # AdamW の Weight Decay
LABEL_SMOOTHING = 0.0    # 例: 0.0 ～ 0.1
DROP_RATE = 0.0          # Classifier Dropout
DROP_PATH_RATE = 0.1     # Stochastic Depth (DropPath)
MAX_GRAD_NORM = 1.0      # 勾配クリッピング（None で無効化）
WARMUP_EPOCHS = 1        # Linear Warmup エポック数（0 で無効化）
USE_SCHEDULER = True     # True: Warmup + CosineAnnealing / False: 固定学習率
USE_AMP = True           # GPU 使用時に Mixed Precision（AMP）を有効化

SAVE_DIR = 'checkpoints_vit'  # モデルのベスト重み保存先ディレクトリ

# --- 4. アンサンブル構成（シード違い・ViT バリアント違いを自由に設定）---
# Dosovitskiy et al. (2020) 準拠の timm 事前学習済み ViT モデル例:
#   'vit_base_patch16_224.augreg_in21k_ft_in1k'   : ViT-B/16 (ImageNet-21K -> 1K, 86.6M params)
#   'vit_base_patch32_224.augreg_in21k_ft_in1k'   : ViT-B/32 (ImageNet-21K -> 1K, 88.2M params, 高速)
#   'vit_large_patch16_224.augreg_in21k_ft_in1k'  : ViT-L/16 (ImageNet-21K -> 1K, 304M params, 大規模)
#   'vit_small_patch16_224.augreg_in21k_ft_in1k'  : ViT-S/16 (ImageNet-21K -> 1K, 22.1M params, 軽量)
#
# ※ まずは 1 モデルだけでパラメータ調整したい場合は、2 番目以降をコメントアウトしてください。
# ※ 各辞書で 'lr', 'epochs', 'weight_decay', 'img_size', 'drop_rate', 'drop_path_rate' を個別指定できます。
ENSEMBLE_CONFIGS = [
    {
        'name': 'vit_b16_seed1234',
        'model_name': 'vit_base_patch16_224.augreg_in21k_ft_in1k',
        'seed': 1234,
        'weight': 1.0,
    },
    {
        'name': 'vit_b16_seed2024',
        'model_name': 'vit_base_patch16_224.augreg_in21k_ft_in1k',
        'seed': 2024,
        'weight': 1.0,
    },
    {
        'name': 'vit_b32_seed3407',
        'model_name': 'vit_base_patch32_224.augreg_in21k_ft_in1k',
        'seed': 3407,
        'weight': 1.0,
    },
]

# --- 5. TTA（Test-Time Augmentation）設定 ---
USE_TTA = True
# 使用可能なビュー:
#   'orig'       : 元画像（リサイズ済み）
#   'hflip'      : 左右反転
#   'zoom'       : 中央拡大クロップ（ズーム）後に再リサイズ
#   'zoom_hflip' : 中央拡大クロップ＋左右反転
TTA_VIEWS = ['orig', 'hflip', 'zoom', 'zoom_hflip']
TTA_ZOOM_CROP_RATIO = 0.875  # 'zoom' 時のクロップ比率（0.85 ～ 0.92 程度が目安）


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = bool(USE_AMP and device.type == 'cuda')
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_scaler = bool(use_amp and amp_dtype == torch.float16)

print(f'Device: {device} | IMAGE_SIZE: {IMAGE_SIZE} | AMP: {use_amp} ({amp_dtype})')
print(f'Ensemble models ({len(ENSEMBLE_CONFIGS)}): {[c["name"] for c in ENSEMBLE_CONFIGS]}')
print(f'TTA: {USE_TTA} | Views: {TTA_VIEWS if USE_TTA else ["orig"]}')

# 2. CIFAR-10 データの読み込みと ViT 向けリサイズ

In [ ]:
ROOT = '.data'
os.makedirs(ROOT, exist_ok=True)
tar_path = os.path.join(ROOT, 'cifar-10-python.tar.gz')
drive_tar_path = '/content/drive/MyDrive/Colab_Data/CIFAR10/cifar-10-python.tar.gz'
mirror_url = 'https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz'

if not os.path.exists(os.path.join(ROOT, 'cifar-10-batches-py')) and (
    not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071
):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        if os.path.exists(drive_tar_path):
            print(f'Copying CIFAR-10 from Google Drive: {drive_tar_path}')
            shutil.copy2(drive_tar_path, tar_path)
    except Exception as e:
        print(f'Google Drive copy skipped ({e})')

    if not os.path.exists(tar_path) or os.path.getsize(tar_path) != 170498071:
        print(f'Downloading CIFAR-10 from mirror: {mirror_url}')
        urllib.request.urlretrieve(mirror_url, tar_path)

raw_train_data = datasets.CIFAR10(root=ROOT, train=True, download=True)

means = raw_train_data.data.mean(axis=(0, 1, 2)) / 255.0
stds = raw_train_data.data.std(axis=(0, 1, 2)) / 255.0

if NORM_MODE == 'cifar10':
    norm_means, norm_stds = means, stds
elif NORM_MODE == 'vit_paper':
    norm_means, norm_stds = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
else:
    raise ValueError(f'Unknown NORM_MODE: {NORM_MODE}. Use "cifar10" or "vit_paper".')

print(f'Calculated CIFAR-10 means: {means}')
print(f'Calculated CIFAR-10 stds : {stds}')
print(f'Active normalization ({NORM_MODE}): mean={norm_means}, std={norm_stds}')

In [ ]:
# 32x32 画像をデータ拡張したのち、ViT 入力解像度 (IMAGE_SIZE x IMAGE_SIZE) へリサイズします
train_transforms = transforms.Compose([
    transforms.RandomRotation(RANDOM_ROTATION),
    transforms.RandomHorizontalFlip(HFLIP_PROB),
    transforms.RandomCrop(32, padding=RANDOM_CROP_PADDING, padding_mode='reflect'),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=RESIZE_INTERPOLATION),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm_means, std=norm_stds),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=RESIZE_INTERPOLATION),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm_means, std=norm_stds),
])

full_train_data = datasets.CIFAR10(
    ROOT,
    train=True,
    download=True,
    transform=train_transforms,
)

test_data = datasets.CIFAR10(
    ROOT,
    train=False,
    download=True,
    transform=test_transforms,
)

# 全アンサンブルモデルで共通の Train / Valid 分割を使用（データリーク防止）
n_train_examples = int(len(full_train_data) * VALID_RATIO)
n_valid_examples = len(full_train_data) - n_train_examples

split_generator = torch.Generator().manual_seed(SEED)
train_data, valid_data = data.random_split(
    full_train_data,
    [n_train_examples, n_valid_examples],
    generator=split_generator,
)

valid_data = copy.deepcopy(valid_data)
valid_data.dataset.transform = test_transforms

classes = test_data.classes
sample_x, sample_y = train_data[0]

print(f'Number of training examples  : {len(train_data)}')
print(f'Number of validation examples: {len(valid_data)}')
print(f'Number of testing examples   : {len(test_data)}')
print(f'Resized sample tensor shape  : {tuple(sample_x.shape)} (class: {classes[sample_y]})')

In [ ]:
def normalize_image(image):
    image = image.clone()
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min=image_min, max=image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image


def plot_images(images, labels, classes, normalize=True):
    n_images = len(images)
    cols = int(math.ceil(math.sqrt(n_images)))
    rows = int(math.ceil(n_images / cols))

    fig = plt.figure(figsize=(10, 10))
    for i in range(n_images):
        ax = fig.add_subplot(rows, cols, i + 1)
        image = images[i]
        if normalize:
            image = normalize_image(image)
        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')
    plt.tight_layout()
    plt.show()


N_IMAGES = 16
preview_images, preview_labels = zip(*[train_data[i] for i in range(N_IMAGES)])
plot_images(preview_images, preview_labels, classes, normalize=True)

# 3. DataLoader の準備

In [ ]:
pin_memory = (device.type == 'cuda')

train_iterator = data.DataLoader(
    train_data,
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

valid_iterator = data.DataLoader(
    valid_data,
    shuffle=False,
    batch_size=EVAL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

test_iterator = data.DataLoader(
    test_data,
    shuffle=False,
    batch_size=EVAL_BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

print(f'Batches -> train: {len(train_iterator)} | valid: {len(valid_iterator)} | test: {len(test_iterator)}')

# 4. Vision Transformer (ViT) モデルの設計

Dosovitskiy et al. (2020) の ViT アーキテクチャは、以下の流れで画像を分類します。

1. **Patch Embedding**: リサイズ済み画像 $(H \times W \times C)$ を $P \times P$ のパッチに分割し、線形射影で埋め込みベクトル列に変換
2. **`[CLS]` Token & Positional Embedding**: 先頭に学習可能なクラス表現トークンを連結し、1D 位置埋め込みを加算（入力解像度を変更した場合は事前学習済み位置埋め込みを 2D 補間）
3. **Transformer Encoder**: Pre-Norm（LayerNorm → Multi-Head Self-Attention → 残差接続 → LayerNorm → GELU MLP → 残差接続）を $L$ 層積み上げ
4. **Classification Head**: 最終層の `[CLS]` トークン表現から 10 クラスのロジットを出力

下の `ViTClassifier` ラッパーは、元の `AlexNet` と同様に `(logits, h)`（ロジットと `[CLS]` 中間表現）を返すため、後半の表現抽出コードとも互換性を保っています。

In [ ]:
class ViTClassifier(nn.Module):
    """Dosovitskiy et al. (2020) Vision Transformer (ViT) ラッパー。

    timm の事前学習済み ViT を読み込み、出力 (logits, h) を返します。
    - img_size を 224 以外（例: 384）に変更した場合も、timm が位置埋め込みを自動で 2D 補間します。
    - 入力テンソルの空間サイズが self.img_size と異なる場合は forward 内で自動リサイズします。
    """

    def __init__(
        self,
        model_name,
        output_dim=10,
        img_size=224,
        pretrained=True,
        drop_rate=0.0,
        drop_path_rate=0.1,
    ):
        super().__init__()
        self.model_name = model_name
        self.img_size = img_size
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=output_dim,
            img_size=img_size,
            drop_rate=drop_rate,
            drop_path_rate=drop_path_rate,
        )

    def forward(self, x):
        if x.shape[-2:] != (self.img_size, self.img_size):
            x = F.interpolate(
                x,
                size=(self.img_size, self.img_size),
                mode='bicubic',
                align_corners=False,
            )
        feats = self.backbone.forward_features(x)
        h = self.backbone.forward_head(feats, pre_logits=True)
        logits = self.backbone.get_classifier()(h)
        return logits, h


def build_vit_model(cfg, output_dim=OUTPUT_DIM, pretrained=True):
    return ViTClassifier(
        model_name=cfg['model_name'],
        output_dim=output_dim,
        img_size=cfg.get('img_size', IMAGE_SIZE),
        pretrained=pretrained,
        drop_rate=cfg.get('drop_rate', DROP_RATE),
        drop_path_rate=cfg.get('drop_path_rate', DROP_PATH_RATE),
    )


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_state_dict_safe(ckpt_path, map_location):
    try:
        return torch.load(ckpt_path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(ckpt_path, map_location=map_location)


# 最初のモデルで構築とパラメータ数を確認（構造確認のみなので pretrained=False で高速チェック）
_preview_cfg = ENSEMBLE_CONFIGS[0]
_preview_model = build_vit_model(_preview_cfg, output_dim=OUTPUT_DIM, pretrained=False)
print(
    f'Preview model: {_preview_cfg["model_name"]} '
    f'(img_size={_preview_cfg.get("img_size", IMAGE_SIZE)}) -> '
    f'{count_parameters(_preview_model):,} trainable parameters'
)
del _preview_model

# 5. 学習・評価・TTA・アンサンブル関数の定義

In [ ]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    return correct.float() / y.shape[0]


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs


def create_warmup_cosine_scheduler(optimizer, epochs, warmup_epochs=1, min_lr_ratio=0.01):
    effective_warmup = min(max(0, warmup_epochs), max(0, epochs - 1))

    def lr_lambda(epoch):
        if effective_warmup > 0 and epoch < effective_warmup:
            return float(epoch + 1) / float(effective_warmup + 1)
        denom = max(1, epochs - effective_warmup)
        progress = min(1.0, float(epoch - effective_warmup) / float(denom))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train(
    model,
    iterator,
    optimizer,
    criterion,
    device,
    scaler=None,
    use_amp=True,
    amp_dtype=torch.float16,
    max_grad_norm=1.0,
):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    amp_device = 'cuda' if device.type == 'cuda' else 'cpu'

    for x, y in tqdm(iterator, desc='Training', leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(amp_device, dtype=amp_dtype, enabled=use_amp):
            y_pred, _ = model(x)
            loss = criterion(y_pred, y)

        if scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        batch_size = y.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (y_pred.argmax(dim=1) == y).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


def evaluate(model, iterator, criterion, device, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    amp_device = 'cuda' if device.type == 'cuda' else 'cpu'

    with torch.no_grad():
        for x, y in tqdm(iterator, desc='Evaluating', leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast(amp_device, dtype=amp_dtype, enabled=use_amp):
                y_pred, _ = model(x)
                loss = criterion(y_pred, y)

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (y_pred.argmax(dim=1) == y).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
def make_tta_views(x, tta_views=('orig', 'hflip', 'zoom', 'zoom_hflip'), zoom_crop_ratio=0.875):
    """バッチ画像 (B, C, H, W) から指定された TTA ビューのリスト [(ビュー名, テンソル), ...] を生成する。"""
    _, _, h, w = x.shape
    views = []

    x_zoom = None
    if any(v in ('zoom', 'zoom_hflip') for v in tta_views):
        crop_h = max(1, int(round(h * zoom_crop_ratio)))
        crop_w = max(1, int(round(w * zoom_crop_ratio)))
        top = (h - crop_h) // 2
        left = (w - crop_w) // 2
        x_cropped = x[:, :, top:top + crop_h, left:left + crop_w]
        x_zoom = F.interpolate(x_cropped, size=(h, w), mode='bicubic', align_corners=False)

    for view_name in tta_views:
        if view_name == 'orig':
            views.append(('orig', x))
        elif view_name == 'hflip':
            views.append(('hflip', torch.flip(x, dims=[3])))
        elif view_name == 'zoom':
            views.append(('zoom', x_zoom))
        elif view_name == 'zoom_hflip':
            views.append(('zoom_hflip', torch.flip(x_zoom, dims=[3])))
        else:
            raise ValueError(
                f'Unknown TTA view: {view_name}. Choose from orig, hflip, zoom, zoom_hflip.'
            )

    return views


def get_predictions(
    model,
    iterator,
    device,
    use_tta=True,
    tta_views=TTA_VIEWS,
    zoom_crop_ratio=TTA_ZOOM_CROP_RATIO,
    use_amp=True,
    amp_dtype=torch.float16,
):
    """単一モデルの予測確率を取得する。

    1 回の推論パスで TTA なし（'orig' が含まれる場合）と TTA ありの両方の確率を返します。
    ※ 224x224 以上の全画像を RAM に溜めないよう、ラベルと確率のみを保持します。
    """
    model.eval()
    active_views = list(tta_views) if use_tta else ['orig']
    if len(active_views) == 0:
        raise ValueError('TTA_VIEWS must contain at least one view when USE_TTA=True.')

    amp_device = 'cuda' if device.type == 'cuda' else 'cpu'
    labels_list = []
    probs_tta_list = []
    probs_orig_list = []
    has_orig = ('orig' in active_views)

    with torch.no_grad():
        for x, y in tqdm(iterator, desc='Predicting', leave=False):
            x = x.to(device, non_blocking=True)
            views = make_tta_views(x, tta_views=active_views, zoom_crop_ratio=zoom_crop_ratio)

            probs_sum = None
            orig_probs = None
            with torch.amp.autocast(amp_device, dtype=amp_dtype, enabled=use_amp):
                for view_name, v in views:
                    logits, _ = model(v)
                    probs = F.softmax(logits.float(), dim=-1)
                    if view_name == 'orig' and orig_probs is None:
                        orig_probs = probs
                    probs_sum = probs if probs_sum is None else (probs_sum + probs)

            probs_tta = probs_sum / len(views)
            labels_list.append(y.cpu())
            probs_tta_list.append(probs_tta.cpu())
            if has_orig:
                probs_orig_list.append(orig_probs.cpu())

    labels = torch.cat(labels_list, dim=0)
    probs_tta = torch.cat(probs_tta_list, dim=0)
    probs_orig = torch.cat(probs_orig_list, dim=0) if has_orig else None
    return labels, probs_orig, probs_tta


def compute_probs_metrics(probs, labels):
    loss = F.nll_loss(torch.log(probs.clamp(min=1e-12)), labels).item()
    acc = (probs.argmax(dim=1) == labels).float().mean().item()
    return loss, acc


def evaluate_ensemble_from_checkpoints(
    trained_entries,
    iterator,
    device,
    use_tta=True,
    tta_views=TTA_VIEWS,
    zoom_crop_ratio=TTA_ZOOM_CROP_RATIO,
    use_amp=True,
    amp_dtype=torch.float16,
    split_name='Test',
):
    """チェックポイントを 1 モデルずつ読み込んで TTA 推論し、重み付きアンサンブル評価を行う。

    GPU メモリ上に複数モデルを同時常駐させないため、モデル数を増やしても省メモリで動作します。
    """
    total_weight = sum(entry['weight'] for entry in trained_entries)
    if total_weight <= 0:
        raise ValueError('Total ensemble weight must be positive.')

    ensemble_probs_orig = None
    ensemble_probs_tta = None
    ref_labels = None
    per_model_results = []

    print(f'=== {split_name} Evaluation (TTA={use_tta}, views={tta_views if use_tta else ["orig"]}) ===')
    for entry in trained_entries:
        cfg = entry['cfg']
        weight = entry['weight']
        model = build_vit_model(cfg, output_dim=OUTPUT_DIM, pretrained=False).to(device)
        model.load_state_dict(load_state_dict_safe(entry['ckpt_path'], map_location=device))

        labels, probs_orig, probs_tta = get_predictions(
            model,
            iterator,
            device,
            use_tta=use_tta,
            tta_views=tta_views,
            zoom_crop_ratio=zoom_crop_ratio,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
        )

        if ref_labels is None:
            ref_labels = labels

        tta_loss, tta_acc = compute_probs_metrics(probs_tta, labels)
        if probs_orig is not None:
            orig_loss, orig_acc = compute_probs_metrics(probs_orig, labels)
            if use_tta:
                print(
                    f"[{entry['name']}] (w={weight}) "
                    f"Single: Loss={orig_loss:.4f}, Acc={orig_acc*100:.2f}% | "
                    f"+TTA: Loss={tta_loss:.4f}, Acc={tta_acc*100:.2f}%"
                )
            else:
                print(
                    f"[{entry['name']}] (w={weight}) "
                    f"Single: Loss={orig_loss:.4f}, Acc={orig_acc*100:.2f}%"
                )
            weighted_orig = probs_orig * (weight / total_weight)
            ensemble_probs_orig = (
                weighted_orig if ensemble_probs_orig is None else (ensemble_probs_orig + weighted_orig)
            )
        else:
            orig_loss, orig_acc = None, None
            print(
                f"[{entry['name']}] (w={weight}) "
                f"+TTA: Loss={tta_loss:.4f}, Acc={tta_acc*100:.2f}%"
            )

        weighted_tta = probs_tta * (weight / total_weight)
        ensemble_probs_tta = (
            weighted_tta if ensemble_probs_tta is None else (ensemble_probs_tta + weighted_tta)
        )

        per_model_results.append({
            'name': entry['name'],
            'weight': weight,
            'orig_loss': orig_loss,
            'orig_acc': orig_acc,
            'tta_loss': tta_loss,
            'tta_acc': tta_acc,
            'probs_orig': probs_orig,
            'probs_tta': probs_tta,
        })

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    print('-' * 72)
    if ensemble_probs_orig is not None and use_tta:
        ens_orig_loss, ens_orig_acc = compute_probs_metrics(ensemble_probs_orig, ref_labels)
        print(
            f'[{split_name} Ensemble (No TTA)] '
            f'Loss: {ens_orig_loss:.4f} | Acc: {ens_orig_acc*100:.2f}%'
        )

    ens_tta_loss, ens_tta_acc = compute_probs_metrics(ensemble_probs_tta, ref_labels)
    label_tag = f'{split_name} Ensemble + TTA' if use_tta else f'{split_name} Ensemble'
    print(f'[{label_tag}] Loss: {ens_tta_loss:.4f} | Acc: {ens_tta_acc*100:.2f}%')

    return ref_labels, ensemble_probs_tta, per_model_results

# 6. 学習（アンサンブル用 ViT モデルのファインチューニング）

`ENSEMBLE_CONFIGS` に登録した各 ViT モデルを順番にファインチューニングし、検証精度（`Val. Acc`）が最も高かった重みを保存します。

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
trained_entries = []
histories = {}

for idx, cfg in enumerate(ENSEMBLE_CONFIGS, start=1):
    run_name = cfg['name']
    run_seed = cfg.get('seed', SEED + idx - 1)
    run_epochs = cfg.get('epochs', EPOCHS)
    run_lr = cfg.get('lr', LR)
    run_wd = cfg.get('weight_decay', WEIGHT_DECAY)
    run_img_size = cfg.get('img_size', IMAGE_SIZE)
    run_weight = cfg.get('weight', 1.0)
    ckpt_path = os.path.join(SAVE_DIR, f'{run_name}_best.pt')

    print(f"\n{'=' * 72}")
    print(f'[{idx}/{len(ENSEMBLE_CONFIGS)}] Training: {run_name}')
    print(
        f"  model={cfg['model_name']} | img_size={run_img_size} | seed={run_seed} | "
        f'epochs={run_epochs} | lr={run_lr:.1e} | wd={run_wd:.1e} | weight={run_weight}'
    )
    print(f"{'=' * 72}")

    set_seed(run_seed)

    # シードごとにシャッフル順序を変えた訓練 DataLoader を作成
    loader_gen = torch.Generator().manual_seed(run_seed)
    run_train_loader = data.DataLoader(
        train_data,
        shuffle=True,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        generator=loader_gen,
    )

    model = build_vit_model(cfg, output_dim=OUTPUT_DIM, pretrained=True).to(device)
    print(f'Trainable parameters: {count_parameters(model):,}')

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
    eval_criterion = nn.CrossEntropyLoss().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=run_lr, weight_decay=run_wd)
    scheduler = (
        create_warmup_cosine_scheduler(optimizer, epochs=run_epochs, warmup_epochs=WARMUP_EPOCHS)
        if USE_SCHEDULER
        else None
    )
    scaler = torch.amp.GradScaler('cuda', enabled=True) if use_scaler else None

    best_valid_acc = 0.0
    best_valid_loss = float('inf')
    history = {'train_loss': [], 'train_acc': [], 'valid_loss': [], 'valid_acc': []}

    for epoch in trange(run_epochs, desc=run_name):
        start_time = time.monotonic()

        train_loss, train_acc = train(
            model,
            run_train_loader,
            optimizer,
            criterion,
            device,
            scaler=scaler,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
            max_grad_norm=MAX_GRAD_NORM,
        )
        valid_loss, valid_acc = evaluate(
            model,
            valid_iterator,
            eval_criterion,
            device,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
        )

        current_lr = optimizer.param_groups[0]['lr']
        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['valid_loss'].append(valid_loss)
        history['valid_acc'].append(valid_acc)

        if (valid_acc > best_valid_acc) or (
            valid_acc == best_valid_acc and valid_loss < best_valid_loss
        ):
            best_valid_acc = valid_acc
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), ckpt_path)

        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        print(
            f'Epoch: {epoch+1:02}/{run_epochs:02} | Time: {epoch_mins}m {epoch_secs}s | '
            f'LR: {current_lr:.2e}'
        )
        print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
        print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

    print(f'Best Val. Acc ({run_name}): {best_valid_acc*100:.2f}% -> saved to {ckpt_path}')
    histories[run_name] = history
    trained_entries.append({
        'name': run_name,
        'cfg': cfg,
        'ckpt_path': ckpt_path,
        'weight': run_weight,
        'best_valid_acc': best_valid_acc,
    })

    del model, optimizer, scheduler, scaler
    if device.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# 学習曲線の可視化（パラメータ調整時の確認用）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for run_name, hist in histories.items():
    epochs_axis = range(1, len(hist['train_loss']) + 1)
    axes[0].plot(epochs_axis, hist['train_loss'], linestyle='--', label=f'{run_name} (Train)')
    axes[0].plot(epochs_axis, hist['valid_loss'], marker='o', label=f'{run_name} (Valid)')
    axes[1].plot(epochs_axis, [a * 100 for a in hist['train_acc']], linestyle='--', label=f'{run_name} (Train)')
    axes[1].plot(epochs_axis, [a * 100 for a in hist['valid_acc']], marker='o', label=f'{run_name} (Valid)')

axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

# 7. 評価（単体 vs TTA vs アンサンブル + TTA）

まず **検証データ（Validation）** でパラメータ・TTA・アンサンブルの効果を確認し、続けて **テストデータ（Test）** で最終評価を行います。

In [ ]:
# 検証データ（Validation）での評価（パラメータやアンサンブル重みの調整はこちらを参考に）
val_labels, val_probs, val_per_model = evaluate_ensemble_from_checkpoints(
    trained_entries,
    valid_iterator,
    device,
    use_tta=USE_TTA,
    tta_views=TTA_VIEWS,
    zoom_crop_ratio=TTA_ZOOM_CROP_RATIO,
    use_amp=use_amp,
    amp_dtype=amp_dtype,
    split_name='Valid',
)

In [ ]:
# テストデータ（Test 10,000枚）での最終評価
labels, probs, test_per_model = evaluate_ensemble_from_checkpoints(
    trained_entries,
    test_iterator,
    device,
    use_tta=USE_TTA,
    tta_views=TTA_VIEWS,
    zoom_crop_ratio=TTA_ZOOM_CROP_RATIO,
    use_amp=use_amp,
    amp_dtype=amp_dtype,
    split_name='Test',
)

test_loss, test_acc = compute_probs_metrics(probs, labels)
print(f'\nFinal Test Loss: {test_loss:.3f} | Final Test Acc: {test_acc*100:.2f}%')

# 8. モデルを説明（混同行列・誤分類画像・特徴表現）

In [ ]:
pred_labels = torch.argmax(probs, dim=1)


def plot_confusion_matrix(labels, pred_labels, classes):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = confusion_matrix(labels, pred_labels)
    cm = ConfusionMatrixDisplay(cm, display_labels=classes)
    cm.plot(values_format='d', cmap='Blues', ax=ax)
    plt.xticks(rotation=20)
    plt.show()


plot_confusion_matrix(labels, pred_labels, classes)

## 学習したモデル（アンサンブル＋TTA）がもっとも自信を持って間違えた画像

In [ ]:
corrects = torch.eq(labels, pred_labels)
incorrect_examples = []

# 全テスト画像を RAM に保持せず、誤分類のインデックスと確率のみを記録
for idx, (label, prob, correct) in enumerate(zip(labels, probs, corrects)):
    if not correct:
        incorrect_examples.append((idx, label.item(), prob))

incorrect_examples.sort(
    reverse=True,
    key=lambda x: torch.max(x[2], dim=0).values.item(),
)
print(f'Total incorrect test predictions: {len(incorrect_examples)} / {len(labels)}')


def plot_most_incorrect(incorrect, dataset, classes, n_images=25, normalize=True):
    if len(incorrect) == 0:
        print('No incorrect predictions found!')
        return

    k = min(n_images, len(incorrect))
    cols = int(math.ceil(math.sqrt(k)))
    rows = int(math.ceil(k / cols))

    fig = plt.figure(figsize=(cols * 4, rows * 4))
    for i in range(k):
        ax = fig.add_subplot(rows, cols, i + 1)
        idx, true_label, prob = incorrect[i]
        image, _ = dataset[idx]
        image = image.permute(1, 2, 0)
        true_prob = prob[true_label]
        incorrect_prob, incorrect_label = torch.max(prob, dim=0)
        true_class = classes[true_label]
        incorrect_class = classes[incorrect_label]

        if normalize:
            image = normalize_image(image)

        ax.imshow(image.cpu().numpy())
        ax.set_title(
            f'true: {true_class} ({true_prob:.3f})\n'
            f'pred: {incorrect_class} ({incorrect_prob:.3f})'
        )
        ax.axis('off')

    fig.subplots_adjust(hspace=0.4)
    plt.show()


N_INCORRECT_IMAGES = 25
plot_most_incorrect(incorrect_examples, test_data, classes, N_INCORRECT_IMAGES)

## 特徴表現（ViT の `[CLS]` トークン表現）の取得

元のノートブックと同様に、分類直前の中間特徴表現（ViT の `[CLS]` トークン出力）を取得できます（PCA / t-SNE などの可視化に利用可能）。

In [ ]:
def get_representations(model, iterator, device, use_amp=True, amp_dtype=torch.float16):
    model.eval()
    outputs = []
    intermediates = []
    labels = []
    amp_device = 'cuda' if device.type == 'cuda' else 'cpu'

    with torch.no_grad():
        for x, y in tqdm(iterator, desc='Extracting representations', leave=False):
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast(amp_device, dtype=amp_dtype, enabled=use_amp):
                y_pred, h = model(x)
            outputs.append(y_pred.float().cpu())
            intermediates.append(h.float().cpu())
            labels.append(y.cpu())

    outputs = torch.cat(outputs, dim=0)
    intermediates = torch.cat(intermediates, dim=0)
    labels = torch.cat(labels, dim=0)
    return outputs, intermediates, labels


# 先頭モデル（trained_entries[0]）を使って検証データの特徴表現を取得
# ※ 必要に応じて valid_iterator を test_iterator や train_iterator に変更してください
repr_entry = trained_entries[0]
repr_model = build_vit_model(repr_entry['cfg'], output_dim=OUTPUT_DIM, pretrained=False).to(device)
repr_model.load_state_dict(load_state_dict_safe(repr_entry['ckpt_path'], map_location=device))

outputs, intermediates, repr_labels = get_representations(
    repr_model, valid_iterator, device, use_amp=use_amp, amp_dtype=amp_dtype
)
print(
    f"Extracted from {repr_entry['name']} -> "
    f'outputs: {tuple(outputs.shape)} | intermediates ([CLS]): {tuple(intermediates.shape)}'
)

del repr_model
if device.type == 'cuda':
    torch.cuda.empty_cache()